 # Whisper Accent Robustness — Model Evaluation

 Compares any set of models on L2-ARCTIC scripted and spontaneous speech.

 Results are cached to CSV — re-run cells freely without re-running inference.

In [1]:
# ── Config ────────────────────────────────────────────────────────────────────
# Add/remove model keys here. Keys must exist in MODEL_REGISTRY below.
MODELS_TO_EVAL  = ["baseline", "baseline_lora", "ctc_aux"]
RESULTS_DIR     = "results/model_perf_comparison"
BATCH_SIZE      = 8

# Set True to skip inference and load cached CSVs (per model per split)
SKIP_IF_CACHED  = True


In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import re
import torch
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from jiwer import wer as jiwer_wer, cer as jiwer_cer

from src.utils.audio_utils import bytes_to_array
from src.utils.load_l2arctic import load_scripted, load_spontaneous, split_dataset
from src.utils.model_loader import load_baseline, load_baseline_lora, load_ctc_aux

os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


Device: cuda


In [3]:
# ── Model registry ─────────────────────────────────────────────────────────────
# To add a new model: add an entry here + a loader function in model_loader.py.
# Each entry: display_name, loader_fn (returns (model, processor))
MODEL_REGISTRY = {
    "baseline":      {"label": "Zero-shot",    "loader": lambda: load_baseline(device=DEVICE)},
    "baseline_lora": {"label": "Naive LoRA FT",      "loader": lambda: load_baseline_lora(device=DEVICE)},
    "ctc_aux":       {"label": "CTC Aux",      "loader": lambda: load_ctc_aux(device=DEVICE)},
}

# Validate requested models
for key in MODELS_TO_EVAL:
    assert key in MODEL_REGISTRY, f"Unknown model key: {key}. Add it to MODEL_REGISTRY."

LABELS = {k: MODEL_REGISTRY[k]["label"] for k in MODELS_TO_EVAL}
print(f"Evaluating: {LABELS}")


Evaluating: {'baseline': 'Zero-shot', 'baseline_lora': 'Naive LoRA FT', 'ctc_aux': 'CTC Aux'}


In [4]:
# ── Shared helpers ────────────────────────────────────────────────────────────

def norm(s):
    if not isinstance(s, str): return ""
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def add_normalized_columns(df, ref_col="text", pred_col="prediction"):
    df = df.copy()
    df["reference_norm"]  = df[ref_col].apply(norm)
    df["prediction_norm"] = df[pred_col].apply(norm)
    return df

def attach_utterance_stats(df):
    df = df.copy()
    df["ref_num_words"] = df["reference_norm"].str.split().str.len()
    df["ref_num_chars"] = df["reference_norm"].str.len()
    return df

def compute_metrics_df(df, ref_col="reference_norm", pred_col="prediction_norm"):
    refs  = df[ref_col].fillna("").tolist()
    preds = df[pred_col].fillna("").tolist()
    return {"wer": jiwer_wer(refs, preds), "cer": jiwer_cer(refs, preds)}

def compute_grouped_metrics(df, group_col="speaker_native_language",
                             ref_col="reference_norm", pred_col="prediction_norm"):
    rows = []
    for grp, sub in df.groupby(group_col):
        refs  = sub[ref_col].fillna("").tolist()
        preds = sub[pred_col].fillna("").tolist()
        rows.append({group_col: grp, "num_utts": len(sub),
                     "wer": jiwer_wer(refs, preds), "cer": jiwer_cer(refs, preds)})
    return pd.DataFrame(rows)

def transcribe(df, processor, model):
    predictions = []
    for start in range(0, len(df), BATCH_SIZE):
        batch_df = df.iloc[start:start + BATCH_SIZE]
        audio_arrays = [bytes_to_array(row["audio"]["bytes"]) for _, row in batch_df.iterrows()]
        inputs = processor(audio_arrays, sampling_rate=16000, return_tensors="pt",
                           truncation=True, return_attention_mask=True)
        with torch.no_grad():
            pred_ids = model.generate(
                inputs.input_features.to(DEVICE),
                attention_mask=inputs.attention_mask.to(DEVICE),
                language="en", task="transcribe",
            )
        predictions.extend(processor.batch_decode(pred_ids, skip_special_tokens=True))
        if (start // BATCH_SIZE) % 10 == 0:
            print(f"  {start}/{len(df)}", end="\r")
    print(f"  {len(df)}/{len(df)} ✓")
    return predictions

def build_results(df, predictions):
    results = df.drop(columns=["audio"]).copy()
    results["prediction"] = predictions
    results = add_normalized_columns(results)
    results = attach_utterance_stats(results)
    results["utt_wer"] = results.apply(
        lambda r: jiwer_wer(r["reference_norm"], r["prediction_norm"])
        if r["reference_norm"] else None, axis=1
    )
    return results

def run_or_load(model_key, split_name, df):
    """Run inference for one model+split, or load from cache."""
    csv_path = f"{RESULTS_DIR}/{model_key}_{split_name}_predictions.csv"
    if SKIP_IF_CACHED and os.path.exists(csv_path):
        print(f"  [cached] {model_key} / {split_name}")
        return pd.read_csv(csv_path)
    print(f"  [running] {model_key} / {split_name} ...")
    model, processor = MODEL_REGISTRY[model_key]["loader"]()
    model.generation_config.suppress_tokens = None
    model.generation_config.begin_suppress_tokens = None
    preds   = transcribe(df, processor, model)
    results = build_results(df, preds)
    results.to_csv(csv_path, index=False)
    del model; torch.cuda.empty_cache()
    return results

print("Helpers loaded.")


Helpers loaded.


In [5]:
# ── Load datasets ─────────────────────────────────────────────────────────────
print("Loading scripted...")
scripted_df = load_scripted()
_, _, scripted_test = split_dataset(scripted_df)
scripted_test = scripted_test.reset_index(drop=True)

print("Loading spontaneous...")
spontaneous_test = load_spontaneous().reset_index(drop=True)

print(f"Scripted test : {len(scripted_test)} utterances")
print(f"Spontaneous   : {len(spontaneous_test)} utterances")


Loading scripted...
Split summary:
  Train :  2294 rows  (63.7%)
  Dev   :   405 rows  (11.3%)
  Test  :   900 rows  (25.0%)  [['BWC', 'EBVS', 'HJK', 'HQTV', 'SKA', 'SVBI']]
Loading spontaneous...
Scripted test : 900 utterances
Spontaneous   : 22 utterances


In [6]:
# ── Run / load all models × both splits ──────────────────────────────────────
results = {}   # results[model_key][split] = DataFrame
for key in MODELS_TO_EVAL:
    results[key] = {
        "scripted":     run_or_load(key, "scripted",     scripted_test),
        "spontaneous":  run_or_load(key, "spontaneous",  spontaneous_test),
    }
print("All inference done.")


  [running] baseline / scripted ...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 15.57 GiB of which 6.00 MiB is free. Process 241474 has 14.03 GiB memory in use. Including non-PyTorch memory, this process has 1.09 GiB memory in use. Of the allocated memory 796.71 MiB is allocated by PyTorch, and 77.29 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

 ---

 # Part 1 — Overall WER / CER Table

In [ ]:
rows = []
for key in MODELS_TO_EVAL:
    for split in ["scripted", "spontaneous"]:
        m = compute_metrics_df(results[key][split])
        rows.append({"Model": LABELS[key], "Split": split.capitalize(),
                     "WER": m["wer"], "CER": m["cer"]})
overall_df = pd.DataFrame(rows)
display(overall_df.style
        .format({"WER": "{:.4f}", "CER": "{:.4f}"})
        .background_gradient(subset=["WER"], cmap="RdYlGn_r")
        .set_caption("Overall WER / CER — All Models"))

overall_df.to_csv(f"{RESULTS_DIR}/overall_summary.csv", index=False)


 ---

 # Part 2 — Per-L1 WER

In [ ]:
def per_l1_all_models(split):
    """Build a wide DataFrame: L1 × model WER columns."""
    dfs = []
    for key in MODELS_TO_EVAL:
        g = compute_grouped_metrics(results[key][split])
        g = g.rename(columns={"wer": f"WER_{key}", "cer": f"CER_{key}"})
        dfs.append(g.set_index("speaker_native_language"))
    wide = pd.concat([d[[c for c in d.columns if c.startswith("WER_")]] for d in dfs], axis=1)
    wide["num_utts"] = dfs[0]["num_utts"]
    wide = wide.reset_index().sort_values(f"WER_{MODELS_TO_EVAL[0]}")

    # Relative improvement over baseline for each model
    base_key = MODELS_TO_EVAL[0]
    for key in MODELS_TO_EVAL[1:]:
        wide[f"rel_imp_{key}"] = (
            (wide[f"WER_{base_key}"] - wide[f"WER_{key}"]) / wide[f"WER_{base_key}"] * 100
        )
    return wide

for split in ["scripted", "spontaneous"]:
    wide = per_l1_all_models(split)
    wide.to_csv(f"{RESULTS_DIR}/per_l1_{split}.csv", index=False)

    fmt = {f"WER_{k}": "{:.4f}" for k in MODELS_TO_EVAL}
    fmt.update({f"rel_imp_{k}": "{:.1f}%" for k in MODELS_TO_EVAL[1:]})
    display(wide.style.format(fmt)
            .background_gradient(subset=[f"WER_{k}" for k in MODELS_TO_EVAL], cmap="RdYlGn_r")
            .set_caption(f"Per-L1 WER — {split.capitalize()}"))


In [ ]:
# ── Per-L1 bar chart (one chart per split) ───────────────────────────────────
COLORS = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f"]

for split in ["scripted", "spontaneous"]:
    wide = per_l1_all_models(split)
    l1s  = wide["speaker_native_language"].tolist()

    fig = go.Figure()
    for i, key in enumerate(MODELS_TO_EVAL):
        fig.add_trace(go.Bar(
            name=LABELS[key], x=l1s, y=wide[f"WER_{key}"].tolist(),
            marker_color=COLORS[i % len(COLORS)],
        ))
    fig.update_layout(
        title=f"WER by L1 — {split.capitalize()}",
        barmode="group",
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
    )
    fig.update_yaxes(title_text="WER", tickformat=".0%")
    fig.update_xaxes(title_text="L1 Background")
    fig.show()


In [ ]:
# ── Relative improvement over baseline (scripted) ────────────────────────────
wide_s = per_l1_all_models("scripted")
l1s = wide_s["speaker_native_language"].tolist()

fig = go.Figure()
for i, key in enumerate(MODELS_TO_EVAL[1:], 1):
    col = f"rel_imp_{key}"
    if col not in wide_s.columns: continue
    colors = ["#5cb85c" if v >= 0 else "#e05c5c" for v in wide_s[col]]
    fig.add_trace(go.Bar(
        name=f"{LABELS[key]} vs {LABELS[MODELS_TO_EVAL[0]]}",
        x=l1s, y=wide_s[col].tolist(),
        marker_color=colors,
        text=[f"{v:+.1f}%" for v in wide_s[col]], textposition="outside",
    ))
fig.update_layout(
    title="Relative WER Improvement over Baseline — Scripted",
    barmode="group",
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
)
fig.update_yaxes(title_text="Rel. Improvement %")
fig.update_xaxes(title_text="L1 Background")
fig.show()


 ---

 # Part 3 — Scripted vs Spontaneous Gap

In [ ]:
fig = go.Figure()
for i, key in enumerate(MODELS_TO_EVAL):
    m_s  = compute_metrics_df(results[key]["scripted"])
    m_sp = compute_metrics_df(results[key]["spontaneous"])
    fig.add_trace(go.Bar(
        name=LABELS[key],
        x=["Scripted", "Spontaneous"],
        y=[m_s["wer"], m_sp["wer"]],
        marker_color=COLORS[i % len(COLORS)],
        text=[f"{m_s['wer']:.3f}", f"{m_sp['wer']:.3f}"], textposition="outside",
    ))
fig.update_layout(
    title="WER: Scripted vs Spontaneous by Model",
    barmode="group",
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
)
fig.update_yaxes(title_text="WER", tickformat=".0%")
fig.update_xaxes(title_text="Speech Type")
fig.show()


 ---

 # Part 4 — Qualitative Examples (Worst Utterances)

In [ ]:
base_key = MODELS_TO_EVAL[0]
for split in ["scripted", "spontaneous"]:
    base_df = results[base_key][split]
    worst   = base_df.nlargest(10, "utt_wer")[
        ["text", "speaker_native_language", "reference_norm", "prediction_norm", "utt_wer"]
    ].rename(columns={
        "prediction_norm": f"pred_{base_key}",
        "utt_wer":         f"wer_{base_key}",
    })
    for key in MODELS_TO_EVAL[1:]:
        worst[f"pred_{key}"] = results[key][split].loc[worst.index, "prediction_norm"].values
        worst[f"wer_{key}"]  = results[key][split].loc[worst.index, "utt_wer"].values
        worst[f"delta_{key}"] = worst[f"wer_{base_key}"] - worst[f"wer_{key}"]

    fmt = {f"wer_{k}": "{:.2f}" for k in MODELS_TO_EVAL}
    fmt.update({f"delta_{k}": "{:+.2f}" for k in MODELS_TO_EVAL[1:]})
    display(worst.sort_values(f"wer_{base_key}", ascending=False)
            .style.format(fmt)
            .set_caption(f"Top-10 Worst ({LABELS[base_key]}) — {split.capitalize()}"))


In [ ]:
# ── Utterance WER distribution ────────────────────────────────────────────────
fig = go.Figure()
for i, key in enumerate(MODELS_TO_EVAL):
    for split, dash in [("scripted", "solid"), ("spontaneous", "dot")]:
        fig.add_trace(go.Histogram(
            x=results[key][split]["utt_wer"],
            name=f"{LABELS[key]} / {split}",
            opacity=0.5, nbinsx=30,
            marker_line_width=0,
        ))
fig.update_layout(
    title="Utterance WER Distribution — All Models & Splits",
    barmode="overlay",
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
)
fig.update_xaxes(title_text="Utterance WER")
fig.update_yaxes(title_text="Count")
fig.show()
